# Library, Utilities, Dataframe Inclusion

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from NLP import *
from itables import show
import itables.options as opt

In [13]:
from pathlib import Path

FOLDER = Path("../GuardianAPI/ArticlesByTopic/GeneralSubtopics/")   # your folder
OUTFILE = FOLDER / "all_topics.csv"

def load_with_source(p: Path) -> pd.DataFrame:
    df = pd.read_csv(p)
    # file base name without extension, e.g. "myokine_topic"
    name = p.stem
    # optional: strip a suffix you added earlier
    if name.endswith("_topic"):
        name = name[:-6]
    # put 'source' as the first column
    df.insert(0, "source", name)
    return df

files = sorted(FOLDER.glob("*.csv"))
dfs = []
for p in files:
    try:
        dfs.append(load_with_source(p))
    except Exception as e:
        print(f"Skipping {p.name}: {e}")

combined = pd.concat(dfs, ignore_index=True, sort=False)
combined.to_csv(OUTFILE, index=False)

print(f"Combined {len(dfs)} files, {combined.shape[0]:,} rows → {OUTFILE}")
print(combined["source"].value_counts().head())

Skipping healthy aging_topic.csv: No columns to parse from file
Combined 8 files, 5,154 rows → ../GuardianAPI/ArticlesByTopic/GeneralSubtopics/all_topics.csv
source
anti-aging     2000
ageing         1524
wellbeing      1266
longevity       235
anti ageing      69
Name: count, dtype: int64


In [16]:
combined = pd.read_csv('../GuardianAPI/ArticlesByTopic/GeneralSubtopics/all_topics.csv')

# Tag Cleaning

In [3]:
tag_clean_df = clean_tags(combined,[])

Kept 331 of 672 articles related to ageing.


# BERTopic Cleaning Full df

In [4]:
topic_model, topic_info, combined_out = analyze_bertopic(combined, v_min_df=1, v_max_df=0.5,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

2025-08-09 16:51:38,440 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████████████████████████████| 21/21 [02:54<00:00,  8.29s/it]
2025-08-09 16:54:32,740 - BERTopic - Embedding - Completed ✓
2025-08-09 16:54:32,741 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-09 16:54:37,575 - BERTopic - Dimensionality - Completed ✓
2025-08-09 16:54:37,576 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-09 16:54:37,602 - BERTopic - Cluster - Completed ✓
2025-08-09 16:54:37,603 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-08-09 16:54:38,731 - BERTopic - Representation - Completed ✓
2025-08-09 16:54:38,732 - BERTopic - Topic reduction - Reducing number of topics
2025-08-09 16:54:38,736 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-09 16:54:39,827 - BERTopic - Representation - Completed ✓
2025-08-09 16:54:39,828 - BERTopic 

In [5]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,126,-1_elizabeth_duke_mps_palace,"[elizabeth, duke, mps, palace, frankland, ikar...","[We’re closing the live blog now, but you can ..."
1,0,116,0_senescent_senescent cells_mice_humans,"[senescent, senescent cells, mice, humans, dna...",[Tech entrepreneur Bryan Johnson is talking ab...
2,1,83,1_mosley_turmeric_calories_gut,"[mosley, turmeric, calories, gut, curcumin, su...",[Chicken soup helps cure colds and flu TRUE “W...
3,2,80,2_aged care_people aged_morrie_betty,"[aged care, people aged, morrie, betty, social...","[On Thursday, the Queen celebrates her 90th bi..."
4,3,73,3_athill_calment_zak_theatre,"[athill, calment, zak, theatre, comedy, keilso...","[Writer and editor Diana Athill, whose clear e..."
5,4,46,4_min_duckett_compton_ashwin,"[min, duckett, compton, ashwin, jurel, innings...",[Arne Slot speaks I don’t know if it was that ...
6,5,32,5_okawa_kimura_world oldest_paternity,"[okawa, kimura, world oldest, paternity, prefe...",[Plenty of sleep and a varied diet are the sec...
7,6,30,6_clinton_abortion_sanders_biden,"[clinton, abortion, sanders, biden, musk, hill...","[President Joe Biden condemned the ruling, cal..."
8,7,28,7_proposed award_lynn_captain tom_mbe,"[proposed award, lynn, captain tom, mbe, vera,...",[Dame Vera Lynn has thwarted plans for a “Vera...
9,8,23,8_infections_virus_reuters_new cases,"[infections, virus, reuters, new cases, new ze...",[This blog is now closed. A summary of key rec...


In [6]:
full_keyword_getter(combined_out)

-1.0 → ['elizabeth', 'duke', 'mps', 'palace', 'frankland', 'ikaria', 'transplant', 'monarch', 'nobel', 'lovelock', 'calne', 'moth', 'sturgeon', 'buettner', 'charles', 'tesco', 'moths', 'dido', 'ikarians', 'majesty', 'rich list', 'nathan', 'leila', 'mensch', 'jubilee', 'help buy', 'friendships', 'lumumba', 'reign', 'edna']
0.0 → ['senescent', 'senescent cells', 'mice', 'humans', 'dna', 'cell', 'rapamycin', 'age related', 'biological', 'autophagy', 'ageing process', 'senescence', 'barzilai', 'steele', 'anti ageing', 'sardinia', 'gps', 'tissue', 'stem', '111', 'cellular', 'samples', 'database', 'supplements', 'heart disease', 'immortality', 'metformin', 'related diseases', 'tiziana', 'newman']
1.0 → ['mosley', 'turmeric', 'calories', 'gut', 'curcumin', 'sugar', 'foods', 'weight loss', 'intermittent', 'intermittent fasting', 'longo', 'bacteria', 'obesity', 'symi', 'protein', 'michael mosley', 'lose weight', 'type diabetes', 'kcal', 'inflammatory', 'lemon', 'juice', 'garlic', 'microbiota', 

In [8]:
combined_out.to_csv('../GuardianAPI/guardian_cleared4.csv', index = False)

In [10]:
show(combined_out[combined_out['topic']==0][['section','headline','trailText', 'tags']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


**Related topics:** 0,1,2,4,5,8

In [10]:
combined_out[combined_out['topic'].isin([-1,0,1,2,4,5,8])]

,source,id,section,pub_date,web_url,headline,trailText,byline,wordcount,bodyText,links,authors,tags,topic,topic_prob,topic_keywords
0,5-2 diet_5 2 diet,media/article/2024/jun/09/michael-mosley-excel...,Media,2024-06-09T13:08:44Z,https://www.theguardian.com/media/article/2024...,Michael Mosley excelled at making complex idea...,"TV presenter, who has died aged 67, skilfully ...",Stuart Heritage,686,Few figures in television changed the way we t...,https://www.theguardian.com/media/2007/feb/08/...,Stuart Heritage,Michael Mosley; Media; Television; Science; Te...,1.0,0.039764,"[mosley, turmeric, calories, gut, curcumin, su..."
1,5-2 diet_5 2 diet,lifeandstyle/2024/feb/12/intermittent-fasting-...,Life and style,2024-02-12T10:00:11Z,https://www.theguardian.com/lifeandstyle/2024/...,"Intermittent fasting: what is it, how does it ...",From the 5:2 diet to Rishi Sunak’s 36 hours of...,Amy Fleming,2121,Until its recent emergence as a mainstream hea...,https://www.nature.com/articles/s42255-023-008...,Amy Fleming,Diets and dieting; Health & wellbeing; Life an...,1.0,0.908459,"[mosley, turmeric, calories, gut, curcumin, su..."
2,5-2 diet_5 2 diet,media/2024/jun/09/dr-michael-mosley-obituary,Media,2024-06-09T14:02:04Z,https://www.theguardian.com/media/2024/jun/09/...,Dr Michael Mosley obituary,Popular celebrity medic who offered health adv...,Sarah Boseley,947,"Dr Michael Mosley, who has died aged 67 on the...",https://www.theguardian.com/profile/mimispence...,Sarah Boseley,Michael Mosley; Diets and dieting; Health; Tel...,1.0,0.063920,"[mosley, turmeric, calories, gut, curcumin, su..."
3,5-2 diet_5 2 diet,media/article/2024/jun/22/michael-mosley-clare...,Media,2024-06-22T10:19:22Z,https://www.theguardian.com/media/article/2024...,Michael Mosley’s wife plans to continue work t...,Dr Clare Bailey posts tribute to ‘adventurous ...,Mabel Banfield-Nwachi,376,The widow of Michael Mosley has posted an emot...,https://www.theguardian.com/uk-news/article/20...,Mabel Banfield-Nwachi,Michael Mosley; Health; Diets and dieting; Med...,1.0,0.018797,"[mosley, turmeric, calories, gut, curcumin, su..."
4,5-2 diet_5 2 diet,media/article/2024/jun/08/michael-mosley-searc...,Media,2024-06-08T13:07:30Z,https://www.theguardian.com/media/article/2024...,Michael Mosley search: what we know so far,"Drones, helicopter and residents aid search fo...",Clea Skopeliti,312,The search for British TV doctor Michael Mosle...,https://www.theguardian.com/media/article/2024...,Clea Skopeliti,Michael Mosley; Greece; UK news; Media; Europe...,1.0,0.018272,"[mosley, turmeric, calories, gut, curcumin, su..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
666,supercentenarian_centenarian,environment/2011/jun/26/moths-britain-varietie...,Environment,2011-06-25T23:05:22Z,https://www.theguardian.com/environment/2011/j...,Britain's moths enjoy their moment in the spot...,"<p>They inspire schoolkids, composers and airc...",Martin Wainwright,1846,Small brown hairy things that thrive after dar...,http://nationalmothnight.info/; http://butterf...,Martin Wainwright,Insects; Butterflies; Wildlife; Zoology; Birdw...,-1.0,0.073485,"[elizabeth, duke, mps, palace, frankland, ikar..."
668,supercentenarian_centenarian,tv-and-radio/2010/feb/03/natural-world-horizon...,Television & radio,2010-02-03T00:08:29Z,https://www.theguardian.com/tv-and-radio/2010/...,Watch this,Natural World | Horizon: Don't Grow Old | Desp...,"Martin Skegg, Will Hodgkinson, Julia Raeside a...",341,"Natural World 8pm, BBC2 With cute creatures po...",NaN,Martin Skegg; Will Hodgkinson; Julia Raeside; ...,Television; Television & radio; Culture,-1.0,0.162763,"[elizabeth, duke, mps, palace, frankland, ikar..."
669,supercentenarian_centenarian,technology/blog/2010/mar/28/times-paywall-chil...,Technology,2010-03-28T20:58:52Z,https://www.theguardian.com/technology/blog/20...,That Times paywall: how young is it expecting ...,<p>Rupert Murdoch's preregistration for the Ti...,Charles Arthur,637,The announcement o

# BERTopic Cleaning Tag Clean DF

In [8]:
show(combined_out[combined_out['topic']==0][['section','headline','trailText', 'tags']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


In [5]:
tag_clean_combined_out = clean_tags(combined_out,[])

Kept 80 of 672 articles related to ageing.


In [7]:
topic_model, topic_info, df_out = analyze_bertopic(tag_clean_df, v_min_df=1, v_max_df=0.7,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

2025-08-09 18:15:15,260 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████████████████████████████| 11/11 [01:28<00:00,  8.04s/it]
2025-08-09 18:16:43,779 - BERTopic - Embedding - Completed ✓
2025-08-09 18:16:43,779 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-09 18:16:43,933 - BERTopic - Dimensionality - Completed ✓
2025-08-09 18:16:43,934 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-09 18:16:43,942 - BERTopic - Cluster - Completed ✓
2025-08-09 18:16:43,943 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-08-09 18:16:44,401 - BERTopic - Representation - Completed ✓
2025-08-09 18:16:44,401 - BERTopic - Topic reduction - Reducing number of topics
2025-08-09 18:16:44,405 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-09 18:16:44,866 - BERTopic - Representation - Completed ✓
2025-08-09 18:16:44,868 - BERTopic 

In [8]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,95,-1_turmeric_curcumin_dna_calment,"[turmeric, curcumin, dna, calment, zak, total,...","[What’s yellow, gathering dust in your cupboar..."
1,0,83,0_senescent_senescent cells_mice_humans,"[senescent, senescent cells, mice, humans, joh...",[Tech entrepreneur Bryan Johnson is talking ab...
2,1,54,1_musk_101_vogue_morrie,"[musk, 101, vogue, morrie, aged care, betty, w...","[On Thursday, the Queen celebrates her 90th bi..."
3,2,53,2_longo_mosley_intermittent fasting_kcal,"[longo, mosley, intermittent fasting, kcal, ty...","[Sad though I am to admit it, the best job in ..."
4,3,18,3_attia_gps_st george_ambulance,"[attia, gps, st george, ambulance, 999, stanle...",[The friend: Alexander Masters I was just putt...
5,4,14,4_physical activity_sedentary_francis_exercises,"[physical activity, sedentary, francis, exerci...","[“Just try,” Francis says. “No.” I dangle list..."
6,5,11,5_okawa_world oldest_kimura_ministry,"[okawa, world oldest, kimura, ministry, prefec...",[The year Jiroemon Kimura was born Bram Stoker...


In [9]:
full_keyword_getter(df_out)

-1.0 → ['turmeric', 'curcumin', 'dna', 'calment', 'zak', 'total', 'humans', 'infections', 'frankland', 'new cases', 'lockdown', 'nobel', 'moths', 'inflammatory', 'jeanne', 'testing', 'lovelock', 'moth', 'pets', 'east', 'transplant', 'microbiota', 'calne', 'investigation', 'authorities', 'rapamycin', 'sardinia', 'prizes', 'owners', 'toole']
0.0 → ['senescent', 'senescent cells', 'mice', 'humans', 'johnson', 'age related', 'autophagy', 'steele', 'senescence', 'rapamycin', 'biological', 'trial', 'newman', 'related diseases', 'cox', 'branyas', 'nobel', 'metformin', 'slow ageing', 'zombie', 'anti ageing', 'menopause', 'senolytics', 'faragher', 'proteins', 'ovaries', 'immortality', 'clinical trial', 'barzilai', 'unity']
1.0 → ['musk', '101', 'vogue', 'morrie', 'aged care', 'betty', 'worker', 'tom', 'design', 'housing', 'fashion', 'rachael', 'veteran', 'federal', 'great grandchildren', 'social care', 'captain', 'bike', 'moore', 'teacher', 'engaged', 'democrats', 'london england', 'captain tom

**Keywords found:**
1. gut microbiota
2. autophagy
3. metformin
4. Senolytics
5. Physical activity / sedentary
6. intermittent-fasting